In [1]:
# 데이터를 분산 처리하고 Spark SQL/입출력을 사용하기 위한 핵심 실행 컨텍스트를 생성합니다.
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

26/02/24 09:02:28 WARN Utils: Your hostname, DESKTOP-G33DRVE resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/24 09:02:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 09:02:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/24 09:02:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/24 09:02:29 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/24 09:02:29 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/02/24 09:02:29 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [3]:
# Parquet 데이터를 로드해 컬럼형 포맷 기반으로 효율적으로 분석합니다.
df_green = spark.read.parquet('data/pq/green/*/*')

In [4]:
# 저수준 변환 및 사용자 정의 집계를 위해 DataFrame을 RDD 흐름으로 전환합니다.
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

In [5]:
# 다음 단계 처리 결과를 만들기 위해 필요한 연산을 수행합니다.
# 시간 기준 필터링에 사용할 datetime 타입을 가져옵니다.
from datetime import datetime

In [7]:
# 시간 조건 필터링 기준을 명확히 하려고 기준 시각을 생성합니다.
# 시간 기반 필터에 사용할 기준 시점을 생성합니다.
start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [8]:
# RDD 샘플을 조회해 변환 전후 레코드 구조를 확인합니다.
rows = rdd.take(10)
row = rows[0]

In [9]:
# 다음 단계 처리 결과를 만들기 위해 필요한 연산을 수행합니다.
# 이 셀은 다음 분석 단계를 위한 중간 결과를 계산합니다.
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 27, 16, 55), PULocationID=123, total_amount=36.01)

In [10]:
# reduceByKey를 위해 행을 (키, 값) 형태로 매핑하는 함수를 정의합니다.
def prepare_for_grouping(row): 
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    
    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

In [11]:
# 같은 키의 매출/건수를 누적 집계하는 reduce 함수를 정의합니다.
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value
    
    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    
    return (output_amount, output_count)

In [12]:
# 집계 결과 필드를 명확히 다루기 위해 구조화된 레코드 타입을 정의합니다.
from collections import namedtuple

In [14]:
# 집계 결과 필드를 명확히 다루기 위해 구조화된 레코드 타입을 정의합니다.
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [15]:
# RDD 결과를 DataFrame 생성용 튜플로 변환하기 위해 언패킹 함수를 정의합니다.
def unwrap(row):
    return RevenueRow(
        hour=row[0][0], 
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )

In [16]:
# 명시적 스키마 정의를 위해 Spark SQL 타입 모듈을 임포트합니다.
from pyspark.sql import types

In [17]:
# 자동 추론 오차를 줄이기 위해 컬럼 타입을 명시적으로 선언합니다.
result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

In [18]:
# RDD 결과를 안전하게 DataFrame으로 변환하려고 결과 스키마를 선언합니다.
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema) 

In [19]:
# 후속 분석 성능을 위해 결과를 Parquet 포맷으로 저장합니다.
df_result.write.parquet('tmp/green-revenue')

26/02/24 09:03:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/24 09:03:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/24 09:03:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/02/24 09:03:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/02/24 09:03:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/02/24 09:03:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/02/24 09:03:22 WARN MemoryManager: Total allocation exceeds 95.

In [20]:
# 저수준 변환 및 사용자 정의 집계를 위해 DataFrame을 RDD 흐름으로 전환합니다.
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .rdd

In [21]:
# 로컬 샘플 검증과 타입 확인을 위해 pandas를 임포트합니다.
import pandas as pd

In [22]:
# RDD 샘플을 조회해 변환 전후 레코드 구조를 확인합니다.
rows = duration_rdd.take(10)

In [23]:
# 샘플 결과를 pandas DataFrame으로 변환해 사람이 읽기 쉽게 확인합니다.
df = pd.DataFrame(rows, columns=columns)

In [24]:
# 다음 단계 처리 결과를 만들기 위해 필요한 연산을 수행합니다.
# 이 셀은 다음 분석 단계를 위한 중간 결과를 계산합니다.
columns

['VendorID',
 'lpep_pickup_datetime',
 'PULocationID',
 'DOLocationID',
 'trip_distance']

In [25]:
#model = ...

# 다음 단계 처리 결과를 만들기 위해 필요한 연산을 수행합니다.
# 이 셀은 다음 분석 단계를 위한 중간 결과를 계산합니다.
def model_predict(df):
#     y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

In [26]:
# 샘플 결과를 pandas DataFrame으로 변환해 사람이 읽기 쉽게 확인합니다.
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions

    for row in df.itertuples():
        yield row

In [27]:
# 다음 단계 처리 결과를 만들기 위해 필요한 연산을 수행합니다.
# 이 셀은 다음 분석 단계를 위한 중간 결과를 계산합니다.
df_predicts = duration_rdd \
    .mapPartitions(apply_model_in_batch)\
    .toDF() \
    .drop('Index')

In [29]:
# 결과 샘플을 눈으로 확인해 변환 로직이 맞는지 검증합니다.
df_predicts.select('predicted_duration').show()

[Stage 8:>                                                          (0 + 1) / 1]

+------------------+
|predicted_duration|
+------------------+
|              21.5|
| 6.949999999999999|
| 5.550000000000001|
|              24.3|
|              9.75|
|              3.35|
|              18.0|
|               2.7|
|               6.0|
|              4.55|
|               0.0|
|3.9000000000000004|
|               5.1|
|               6.0|
|               5.0|
|               2.2|
|              36.6|
|              5.85|
|              10.8|
|              10.3|
+------------------+
only showing top 20 rows

